# Capítulo 3 — Ajustar un modelo · Solución

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch03_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada respuesta empieza con el razonamiento en papel y luego muestra su equivalente en Python.

**Pérdida utilizada**: el error cuadrático medio (ecuación 3.2) en las viviendas A a F, salvo que se indique lo contrario.

**Contenido.** Datos · Los cálculos de las lecciones 3.1 a 3.6 · Ejercicios 3.1 a 3.6

*La celda de la lección 3.4 y la del ejercicio 3.4 f) encadenan cientos de miles de pasos: cuenta unos segundos para cada una.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel (observado, predicho, desviación, recta del colega)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Datos

El identificador D0.1 es el del libro y del archivo CSV.

In [ ]:
# Datos del capítulo 3, copiados del archivo CSV del libro (no modificar)

# D0.1 — las seis viviendas de la introducción, usadas para el ajuste
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)

# D3.1 — cuatro ventas nuevas, usadas solo para evaluar (lección 3.6)
new_sales = pd.DataFrame({
    'sale':      ['V1', 'V2', 'V3', 'V4'],
    'floor_area': [38, 58, 76, 92],
    'price':  [132, 215, 254, 322],
})
xv = new_sales['floor_area'].to_numpy(dtype=float)
yv = new_sales['price'].to_numpy(dtype=float)

# Vivienda G (introducción): ejemplo de cálculo, nunca usada para ajustar
x_G, y_G = 62, 209

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `home` | identificador de la vivienda (A, B…) | — | `logement` |
| `floor_area` | superficie habitable de la vivienda | m² | `surface_m2` |
| `price` | precio de venta | k€ | `prix_kEUR` |
| `sale` | identificador de la venta nueva (V1 a V4) | — | `vente` |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 3.1 a 3.6.

In [ ]:
# Lección 3.1 — dos rectas, dos medidas de error (ecuaciones 3.1 y 3.2)
def mae(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for name, a, b in [("introducción", 3, 30), ('colega', 3.3, 14)]:
    print(f'{name:10s} EAM = {mae(a, b):.2f} k€   ECM = {mse(a, b):.2f} k€²   RECM = {np.sqrt(mse(a, b)):.2f} k€')

In [ ]:
# Lección 3.2 — la curva de pérdida, b fijado en 30 k€ (ecuación 3.3)
slopes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(mse(a, 30), 2) for a in slopes], index=slopes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [mse(a, 30) for a in A], color=BLUE)
plt.scatter(slopes, [mse(a, 30) for a in slopes], color='black', zorder=3)
plt.xlabel('Pendiente a (k€ por m²)'); plt.ylabel('Pérdida L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Lección 3.3 — secantes, derivada (ecuación 3.4) y dos actualizaciones (ecuación 3.5)
def derivative_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'secante h = {h}: {(mse(2.9 + h, 30) - mse(2.9, 30)) / h:.1f}')
print('derivada exacta en 2,9:', round(derivative_a(2.9), 1))

a, eta = 2.5, 0.00006
for stage in range(3):
    print(f'paso {stage}: a = {a}, L = {mse(a, 30):.2f}, L\'(a) = {derivative_a(a):.2f}')
    a = round(a - eta * derivative_a(a), 3)    # redondeado a la milésima, como en el libro

In [ ]:
# Lección 3.4 — el gradiente (ecuación 3.6) y un paso sobre los dos coeficientes (ecuación 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('gradiente en (2,5 ; 30):', g.round(2))
print('tras un paso:', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'tras {k} pasos: a = {a:.4f}, b = {b:.4f}, L = {mse(a, b):.2f}')
a_ls, b_ls = np.polyfit(x, y, 1)
print(f'fondo del valle: a = {a_ls:.3f}, b = {b_ls:.2f}, L = {mse(a_ls, b_ls):.2f}')
print(f'G, ejemplo de cálculo: {a_ls * x_G + b_ls:.0f} k€ predichos para {y_G} k€ observados')

In [ ]:
# Lección 3.5 — dos descensos desde las mismas predicciones (ecuación 3.8)
mu, sigma = x.mean(), x.std()          # std() divide entre n, como en el capítulo 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descent_m2(eta, n_steps, a=2.5, b=30.0):
    for _ in range(n_steps):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descent_std(eta, n_steps, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # ecuación 2.7: mismas predicciones al inicio
    for _ in range(n_steps):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # ecuación 3.8: vuelta a (a ; b)

print('pasos    | m², η = 0,000 06           | estandarizadas, η = 0,3')
a, b = 2.5, 30.0
done = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descent_m2(0.00006, k - done, a, b); done = k
    a_s, b_s = descent_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {mse(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {mse(a_s, b_s):8.2f}')

# Tasas límite: más allá, el descenso diverge (unos 0,000 22 en m², 1 estandarizado)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta}: pérdida tras 400 pasos = {mse(*descent_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'estandarizadas, η = {eta}: pérdida tras 400 pasos = {mse(*descent_std(eta, 400)):.4g} k€²')

In [ ]:
# Lección 3.6 — recta de mínimos cuadrados (ecuación 3.9) y R² (ecuación 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_ls = cov / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_ls:.3f} k€/m², b = {b_ls:.2f} k€')

def r2_lesson(observed, predicted, reference):
    """R² = 1 − ECM de la regla / ECM de la predicción constante « reference »."""
    return 1 - np.mean((observed - predicted) ** 2) / np.mean((observed - reference) ** 2)

y_mean = y.mean()                      # precio medio de A a F: la referencia del R² en A a F
print(f'A a F: ECM constante = {np.mean((y - y_mean) ** 2):.2f}, ECM recta = {mse(a_ls, b_ls):.2f}, '
      f'R² = {r2_lesson(y, a_ls * x + b_ls, y_mean):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# Las tres rectas, la misma medida, en A a F y luego en V1 a V4 (lección 3.6)
lines = {'mínimos cuadrados': (a_ls, b_ls), 'introducción': (3, 30), 'colega': (3.3, 14)}
for name, (a, b) in lines.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{name:16s} A a F: ECM = {np.mean(e_af ** 2):6.2f}  EAM = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 a V4: ECM = {np.mean(e_v ** 2):6.2f}  EAM = {np.mean(np.abs(e_v)):5.2f}   desviaciones {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # si V4 se hubiera vendido a 312 k€
for name in ('mínimos cuadrados', 'colega'):
    a, b = lines[name]
    print(f'V4 a 312 k€: {name} ECM = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, ejemplo de cálculo: {a_ls * x_G + b_ls:.2f} k€ predichos, desviación {y_G - (a_ls * x_G + b_ls):.2f} k€')

## Ejercicio 3.1 — Leer un gráfico: dos medidas, dos mínimos

*Datos D0.1. Nociones: EAM y ECM (lección 3.1), curva de pérdida (lección 3.2), mediana (lección 1.3).* La pendiente está fijada en 3 k€/m². Responde primero a las preguntas a) a e) leyendo la figura del libro.

### a) a e) Solución

**a)** El ECM es mínimo hacia $b \approx 30$ k€ y vale entonces unos 57 k€². El cálculo de f) da exactamente $b = 30{,}33$ y 57,22 k€².

**b)** La curva del EAM está formada por segmentos de recta. Alrededor de su mínimo forma una **meseta** horizontal, de $b = 24$ a $b = 35$ k€, a una altura de unos 7,3 k€. Todos estos valores de $b$ dan el mismo EAM.

**c)** Para $b = 30$, se lee un ECM de unos 57 k€² y un EAM de unos 7,3 k€. La tabla de desviaciones de la lección 3.1 da 57,33 k€² y 7,33 k€ para la recta de la introducción: la lectura es coherente.

**d)** Con $b = 25$, las desviaciones valen $r_i - 25$: −3, 12, −2, 10, −1 y 16 k€. Sus valores absolutos suman 44, como para $b = 30$: el EAM sigue valiendo 44 / 6 ≈ 7,33 k€. Según el EAM, las dos rectas son **equivalentes**. Sus cuadrados suman 9 + 144 + 4 + 100 + 1 + 256 = 514, de donde un ECM de 514 / 6 ≈ 85,67 k€², frente a 57,33: según el ECM, la recta del colega es **peor**. El veredicto depende de la medida, como en la lección 3.1.

**e)** En la meseta, la pendiente de la curva vale 0. Un descenso de gradiente que llega allí se detiene donde está: no puede elegir entre 24 y 35 k€. En las esquinas, la pendiente cambia bruscamente y la derivada no existe. La curva del ECM es una parábola suave: tiene un único punto más bajo, y su pendiente, proporcional a la distancia a ese punto, indica en cada paso en qué sentido ir y cuánto.

### f) En el notebook

In [ ]:
def mse_b(b):
    return np.mean((y - (3 * x + b)) ** 2)

def mae_b(b):
    return np.mean(np.abs(y - (3 * x + b)))

In [ ]:
# Comprobación y luego gráfico de las dos curvas
if mse_b(30) is None or mae_b(30) is None:
    print('Completa mse_b y mae_b.')
else:
    print('mse_b y mae_b:', 'correcto' if np.isclose(mse_b(30), 344 / 6) and np.isclose(mae_b(30), 44 / 6) else 'revisar')
    B = np.linspace(15, 45, 601)
    fig, (h, low) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [mse_b(b) for b in B], color=BLUE); h.set_ylabel('ECM (k€²)'); h.grid(alpha=0.4)
    low.plot(B, [mae_b(b) for b in B], color=ORANGE); low.set_ylabel('EAM (k€)'); low.grid(alpha=0.4)
    low.set_xlabel('Ordenada en el origen b (k€)')
    plt.show()
    print('esquinas de la curva del EAM (r_i ordenados):', np.sort(y - 3 * x))

In [ ]:
r_sorted = np.sort(y - 3 * x)                    # 22, 23, 24, 35, 37, 41
median_r = (r_sorted[2] + r_sorted[3]) / 2       # n = 6 par: posiciones 3 y 4 → 29,5
mean_r = np.mean(y - 3 * x)                  # 182 / 6 ≈ 30,33

In [ ]:
# Comprobación
for name, yours, truth in [('mediana', median_r, 29.5), ('media', mean_r, 182 / 6)]:
    print(f'{name} de los r:', 'aún sin completar' if yours is None else ('correcto' if np.isclose(yours, truth) else 'revisar'))
if median_r is not None and mae_b(30) is not None:
    print(f'EAM en la mediana: {mae_b(median_r):.2f} k€ (en la meseta); '
          f'ECM en la media: {mse_b(mean_r):.2f} k€² (mínimo)')

### Solución de f)

Las esquinas están en 22, 23, 24, 35, 37 y 41 k€, los seis valores $r_i$. La meseta va del tercer al cuarto valor ordenado. La mediana de los $r_i$, $(24 + 35)/2 = 29{,}5$ k€, está en ella. El mínimo del ECM se alcanza en la media de los $r_i$, 30,33 k€. Volvemos al capítulo 1: la media responde al ECM y la mediana al EAM. La mediana resiste a los valores extremos, la media no. Del mismo modo, el ECM penaliza más los errores grandes que el EAM.

## Ejercicio 3.2 — Dos pasos de descenso de gradiente

*Datos D0.1. Nociones: error cuadrático medio (lección 3.1), derivada y actualización (lección 3.3).* La pendiente está fijada en 3 k€/m². Buscamos la ordenada en el origen $b$. Haz primero las preguntas a) a d) en papel.

### a) Preparar

In [ ]:
r = y - 3 * x
r

In [ ]:
# Comprobación
if r is None:
    print('Completa la celda anterior.')
else:
    print('r:', 'correcto' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'revisar')

### a) a c) Solución en papel

**a)** $r = (22 ; 37 ; 23 ; 35 ; 24 ; 41)$ k€. Por ejemplo, para A: 118 − 3 × 32 = 118 − 96 = 22. La desviación vale entonces $e_i = y_i - 3x_i - b = r_i - b$.

**b) y c)** Con $\eta = 0{,}3$, la actualización vale $b - 0{,}3 \times \left(-\frac{2}{6}\sum e_i\right) = b + 0{,}1 \sum e_i$.

| Paso | $b$ (k€) | Desviaciones $r_i - b$ (k€) | $\sum e_i$ | $L(b)$ (k€²) | $L'(b)$ | Avance |
|:--:|--:|:--|--:|--:|--:|--:|
| 0 | 20 | 2 ; 17 ; 3 ; 15 ; 4 ; 21 | 62 | 984 / 6 = 164 | −20,67 | +6,2 |
| 1 | 26,2 | −4,2 ; 10,8 ; −3,2 ; 8,8 ; −2,2 ; 14,8 | 24,8 | 445,84 / 6 ≈ 74,31 | −8,27 | +2,48 |
| 2 | 28,68 | −6,68 ; 8,32 ; −5,68 ; 6,32 ; −4,68 ; 12,32 | 9,92 | ≈ 59,96 | −3,31 | |

Detalle del paso 0: los cuadrados valen 4, 289, 9, 225, 16 y 441, con una suma de 984. La derivada vale $-\frac{2}{6} \times 62 \approx -20{,}67$. El avance vale $0{,}3 \times 20{,}67 = 6{,}2$, y $b_1 = 20 + 6{,}2 = 26{,}2$.

In [ ]:
L_b0 = 164
derivative_b0 = -20.67
b1 = 26.2
L_b1 = 74.31
b2 = 28.68
L_b2 = 59.96

In [ ]:
# Comprobación (cálculo independiente)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
expected_values = {'L_b0': (L_b0, _L(20)), 'derivative_b0': (derivative_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for name, (yours, truth) in expected_values.items():
    status = 'aún sin completar' if yours is None else ('correcto' if abs(yours - truth) < 0.01 else 'revisar')
    print(f'{name}: {status}')

### d) Interpretar

La pérdida disminuye en cada paso: 164, luego 74,31 y luego 59,96 k€². Los pasos se acortan: 6,2 y luego 2,48. Con $\eta = 0{,}3$, cada paso reduce en un 60 % la distancia al mínimo. En efecto, $b_{\text{nuevo}} = b + 0{,}1 \times 6\,(\bar{r} - b)$, así que $\bar{r} - b$ se multiplica por 0,4. El descenso se dirige hacia $\bar{r} = 182/6 \approx 30{,}33$ k€, la **media de los $r_i$**. En el mínimo, la pérdida vale la varianza de los $r_i$, unos 57,22 k€², es decir $7{,}56^2$: es la desviación típica de las desviaciones de la recta de la introducción (ejercicio 1.6).

### e) En el notebook

In [ ]:
def loss(b):
    """ECM de la recta y = 3x + b en A a F (ecuación 3.2)."""
    e = y - (3 * x + b)
    return np.mean(e ** 2)

def derivative(b):
    """L'(b) = −(2/n) Σ e_i (ecuación 3.6)."""
    e = y - (3 * x + b)
    return -2 / len(x) * np.sum(e)

Las dos funciones retoman al pie de la letra las ecuaciones 3.2 y 3.6, con una pendiente fijada en 3.

In [ ]:
def descent(b0, eta, n_steps):
    """Devuelve la lista de los b y la lista de las pérdidas, del paso 0 al paso n_steps."""
    bs, losses = [b0], [loss(b0)]
    for _ in range(n_steps):
        bs.append(bs[-1] - eta * derivative(bs[-1]))   # ecuación 3.5
        losses.append(loss(bs[-1]))
    return bs, losses

if loss(20) is None or derivative(20) is None:
    print('Completa loss y derivative.')
else:
    bs, losses = descent(20, 0.3, 2)
    print('dos pasos:', np.round(bs, 3), np.round(losses, 2), '→',
          'correcto' if np.allclose(bs, [20, 26.2, 28.68]) else 'revisar')
    bs, losses = descent(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(losses, marker='o', color=BLUE)
    plt.xlabel('Paso'); plt.ylabel('Pérdida (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b tras 20 pasos:', round(bs[-1], 3), '  media de los r:', round(np.mean(y - 3 * x), 3))

### f) Predecir y luego comprobar: $\eta = 1{,}2$

**Predicción.** Con $\eta = 1{,}2$, la actualización multiplica la distancia al mínimo por $1 - 2\eta = -1{,}4$. Cada paso sobrepasa el mínimo y cae más lejos del otro lado: $b$ oscila a ambos lados de 30,33 alejándose de él, y la pérdida aumenta. El descenso **diverge**: 20 → 44,8 → 10,08 → 58,69 → −9,36… Una tasa demasiado grande no ralentiza el descenso: lo hace fracasar.

In [ ]:
if loss(20) is not None and derivative(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, losses = descent(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLUE)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Paso'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Ejemplo de cálculo con G

In [ ]:
price_G = 3 * x_G + 28.68     # = 186 + 28,68 = 214,68 k€
gap_G = y_G - price_G       # = 209 − 214,68 = −5,68 k€

In [ ]:
# Comprobación
if price_G is None or gap_G is None:
    print('Completa la celda anterior.')
else:
    print('G:', 'correcto' if np.isclose(price_G, 214.68) and np.isclose(gap_G, -5.68) else 'revisar')

### Solución de g)

Precio predicho: 3 × 62 + 28,68 = 186 + 28,68 = **214,68 k€**. Desviación con signo: 209 − 214,68 = **−5,68 k€**. G no se ha usado para ajustar $b$. Este cálculo muestra cómo usar la regla obtenida tras dos pasos, pero no mide su calidad en ventas nuevas: una sola vivienda no lo permitiría, y el capítulo 6 tratará esta cuestión.

## Ejercicio 3.3 — ¿Qué significa «99 %»?

*Datos D0.1 y D3.1. Nociones: coeficiente de determinación, ajuste y evaluación (lección 3.6).* Haz primero las preguntas a) a f) en papel.

### a) a f) Solución

**a)** Con la ecuación 3.10 y los ECM de la lección 3.1: recta de la introducción, $R^2 = 1 - 57{,}33 / 5\,205{,}22 \approx 0{,}989$; recta del colega, $R^2 = 1 - 65{,}11 / 5\,205{,}22 \approx 0{,}987$. La referencia es la predicción constante igual al precio medio de A a F, 222,33 k€, y los dos ECM se calculan en A a F.

**b)** El denominador, el ECM de la predicción constante en A a F, es el mismo para todas las rectas. Por tanto, el $R^2$ es tanto mayor cuanto menor es el ECM. Y la recta de mínimos cuadrados es, por definición, aquella cuyo ECM en A a F es el más pequeño. Ninguna recta puede tener un $R^2$ mejor en estas seis ventas.

**c)** Precios predichos: 3 × 38 + 30 = 144, 3 × 58 + 30 = 204, 3 × 76 + 30 = 258 y 3 × 92 + 30 = 306 k€. Desviaciones: 132 − 144 = −12, 215 − 204 = +11, 254 − 258 = −4 y 322 − 306 = +16 k€. Cuadrados: 144, 121, 16 y 256, con una suma de 537. ECM: 537 / 4 = 134,25 k€².

**d)** En A a F: mínimos cuadrados (45,36), introducción (57,33), colega (65,11). En V1 a V4: colega (70,73), mínimos cuadrados (93,29), introducción (134,25). La inversión no prueba que la recta del colega sea mejor. Las tres rectas están cerca y cuatro ventas son muy pocas: con V4 vendida a 312 k€ en lugar de 322, la recta de mínimos cuadrados volvería a ir delante (61,11 frente a 73,73 k€²). Solo muestra que la clasificación obtenida durante el ajuste no se traslada tal cual a ventas nuevas.

**e)** Por ejemplo: «En las seis ventas que nos sirvieron para trazar nuestra recta, reduce en un 98,9 % el error cuadrático medio de la predicción constante igual a su precio medio, 222,33 k€. En cuatro ventas nuevas, su error cuadrático medio vale 134,25 k€², es decir, unos 11,6 k€.» Conviene evitar la palabra «explica»: el $R^2$ compara dos errores, no mide una causa.

**f)** Sería rehacer en V1 a V4 lo que el ajuste hacía en A a F. En cuanto se usan estas ventas para elegir, dejan de ser nuevas para la recta elegida: su ECM en V1 a V4 se convierte a su vez en un error de selección, optimista. Para anunciar un error honesto, harían falta otras ventas, nunca usadas ni para ajustar ni para elegir. El capítulo 6 organizará esta división.

### g) En el notebook

In [ ]:
def sales_mse(observed, predicted):
    """observed y predicted: arrays de numpy de la misma longitud."""
    return np.mean((observed - predicted) ** 2)

In [ ]:
# Comprobación: la clasificación de la lección 3.6 y luego el R² en A a F (referencia: precio medio de A a F)
if sales_mse(y, y) is None:
    print('Completa la función sales_mse.')
else:
    a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_ls = y.mean() - a_ls * x.mean()
    expected_values = {'mínimos cuadrados': ((a_ls, b_ls), 45.36, 93.29), 'introducción': ((3, 30), 57.33, 134.25),
                'colega': ((3.3, 14), 65.11, 70.73)}
    for name, ((a, b), att_af, att_v) in expected_values.items():
        e1, e2 = sales_mse(y, a * x + b), sales_mse(yv, a * xv + b)
        print(f'{name:16s} A a F: {e1:6.2f}   V1 a V4: {e2:6.2f} →',
              'correcto' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'revisar')
    r2_af = 1 - sales_mse(y, a_ls * x + b_ls) / sales_mse(y, np.full(6, y.mean()))
    print(f'R² en A a F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'correcto' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'revisar')

## Ejercicio 3.4 — Un descenso con superficies estandarizadas

*Datos D0.1. Nociones: gradiente (lección 3.4), estandarización y reconversión (lección 3.5), recta de mínimos cuadrados (lección 3.6).*

### a) Solución

Con $(w ; c) = (56{,}99 ; 190)$: $56{,}99 \times 1{,}4915 + 190 \approx 85{,}00 + 190 = 275{,}0$ k€. Con $(a ; b) = (2{,}5 ; 30)$: $2{,}5 \times 98 + 30 = 245 + 30 = 275$ k€. Las dos escrituras dan la misma predicción para F, como anuncia la ecuación 2.7.

### b) Media, desviación típica y puntuaciones estandarizadas

In [ ]:
mu = x.mean()            # 64 m²
sigma = x.std()          # 22,796 m²: np.std divide entre n por defecto
z = (x - mu) / sigma
z.round(4)

In [ ]:
# Comprobación
if z is None:
    print('Completa la celda anterior.')
else:
    print('puntuaciones z:', 'correcto' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'revisar')

### c) El gradiente en la escritura estandarizada

In [ ]:
def standardized_gradient(w, c):
    e = y - (w * z + c)
    return np.array([-2 / len(x) * np.sum(z * e), -2 / len(x) * np.sum(e)])

In [ ]:
# Comprobación en el punto de partida (mismas predicciones que (2,5 ; 30))
if z is None or standardized_gradient(1, 1) is None:
    print('Completa las celdas b) y c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = standardized_gradient(w0, c0)
    print(f'inicio: w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'correcto' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'revisar')

### d) Treinta pasos y luego vuelta a las unidades de origen

In [ ]:
w, c = 2.5 * sigma, 30 + 2.5 * mu
for _ in range(30):
    w, c = np.array([w, c]) - 0.3 * standardized_gradient(w, c)
a_std = w / sigma
b_std = c - a_std * mu
print(f'w = {w:.3f}, c = {c:.3f}  →  a = {a_std:.3f} k€/m², b = {b_std:.2f} k€')

In [ ]:
# Comprobación: ecuación 3.9 y np.polyfit
a_ls = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_ls = y.mean() - a_ls * x.mean()
print(f'ecuación 3.9: a = {a_ls:.3f}, b = {b_ls:.2f}; np.polyfit: {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Completa la celda anterior.')
else:
    print('recta obtenida:', 'correcto' if np.allclose([a_std, b_std], [a_ls, b_ls], atol=1e-4) else 'revisar')

### e) y f) La pérdida de los dos descensos y luego el número de pasos

In [ ]:
# Los dos descensos de la lección 3.5, paso a paso (nada que completar)
def path_m2(n_steps, eta=0.00006):
    a, b, losses = 2.5, 30.0, []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return losses

def path_std(n_steps, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, losses = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(n_steps + 1):
        losses.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return losses

plt.figure(figsize=(6, 3))
plt.semilogy(path_m2(30), marker='o', color=ORANGE, label='superficies en m², η = 0,000 06')
plt.semilogy(path_std(30), marker='s', color=BLUE, label='superficies estandarizadas, η = 0,3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Número de pasos'); plt.ylabel('Pérdida (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Número de pasos para quedar a menos de 0,1 k€² del mínimo (unos segundos)
L_min = np.mean((y - (a_ls * x + b_ls)) ** 2)

def n_steps(X, a, b, eta, limit=10 ** 6):
    for k in range(1, limit + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

losses_std = path_std(30)
print('superficies estandarizadas, η = 0,3:', next(k for k, v in enumerate(losses_std) if v - L_min < 0.1), 'pasos')
print('superficies en m², η = 0,000 06:    ', n_steps(x, 2.5, 30.0, 0.00006), 'pasos')

### Solución de f)

El descenso estandarizado llega a menos de 0,1 k€² del mínimo en **6 pasos**, el descenso en m² en **169 808 pasos**. Los dos encuentran la misma recta, 3,151 × superficie + 20,67. Para recordar: antes de un descenso de gradiente, se estandarizan las columnas con escalas muy diferentes. Para leer los coeficientes en las unidades de origen, se reconvierten después con la ecuación 3.8.

## Ejercicio 3.5 — Predecir y luego comprobar: superficies en decenas de m²

*Datos D0.1. Nociones: cambio de escala (lección 2.5), gradiente (lección 3.4), estandarización (lección 3.5).*

### a) a d) Predicciones esperadas

**a)** La recta no cambia: son las mismas viviendas y los mismos precios, solo cambia la unidad de la columna. Una decena de m² son 10 m², así que la pendiente se multiplica por 10: 31,51 k€ por decena de m². La ordenada en el origen, en k€, sigue siendo 20,67.

**b)** El ajuste de partida pasa a ser (25 ; 30). Las desviaciones son las mismas, y cada $x_i$ se divide entre 10. El cociente vale, por tanto, 74,5 / 10 ≈ 7,45 decenas de m², es decir, siempre 74,5 m²: la misma superficie, expresada en otra unidad.

**c)** Mayor. La derivada respecto a la pendiente se divide entre 10, y un mismo desplazamiento de la recta requiere una pendiente 10 veces mayor. La tasa límite se multiplica, por tanto, por unos 10 × 10 = 100: pasa de unos 0,000 22 a unos 0,021.

**d)** Menos pasos que el descenso en m², porque los pasos pueden ser 100 veces mayores. Pero más que el descenso estandarizado: las superficies no están centradas, y el valle sigue inclinado.

### e) Comprobar

In [ ]:
x_tens = x / 10
x_tens

In [ ]:
# Comprobación y luego las medidas (nada más que completar)
if x_tens is None:
    print('Completa la celda anterior.')
else:
    print('x_tens:', 'correcto' if np.allclose(x_tens, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'revisar')
    print('a) recta de mínimos cuadrados (pendiente, ordenada):', np.round(np.polyfit(x_tens, y, 1), 2))
    e0 = y - (25 * x_tens + 30)
    print(f'b) cociente en el paso 0: {np.sum(x_tens * e0) / np.sum(e0):.2f} decenas de m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_tens - b
            a, b = a + eta * 2 / 6 * np.sum(x_tens * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta}: pérdida tras 200 pasos = {np.mean((y - a * x_tens - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_tens - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_tens * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_tens - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0,006: {k} pasos para quedar a menos de 0,1 k€² del mínimo')

### e) y f) Solución

El notebook confirma las predicciones: pendiente 31,51 k€ por decena de m², ordenada 20,67 k€; cociente 7,45; con 0,021 el descenso converge, con 0,022 diverge; con $\eta = 0{,}006$ hacen falta **1785 pasos**, frente a 169 808 para las superficies en m² y 6 para las superficies estandarizadas (ejercicio 3.4).

Cambiar de unidad no ha cambiado la recta óptima: 31,51 k€ por decena de m² son 3,151 k€ por m². Ha cambiado el trayecto y la tasa utilizable. La estandarización va más lejos. Divide entre la desviación típica, lo que pone los dos coeficientes en la misma escala. También resta la media: con $\sum z_i = 0$, la pendiente y la ordenada en el origen ya no se estorban, y el valle deja de estar inclinado (ejercicio 3.6).

## Ejercicio 3.6 — Desafío: círculos en el mapa

*Datos D0.1. Nociones: estandarización (lecciones 1.4 y 3.5), derivadas parciales y gradiente (lección 3.4).* Las preguntas a) a f) se hacen en papel. El notebook comprueba tus resultados.

### a) a f) Solución

**a)** $\sum z_i = \sum (x_i - \mu)/\sigma = (\sum x_i - n\mu)/\sigma = 0$, ya que $n\mu = \sum x_i$. Y $\sum z_i^2 = \sum (x_i - \mu)^2 / \sigma^2 = n\sigma^2 / \sigma^2 = n$, ya que la desviación típica divide entre $n$.

**b)** $\partial L / \partial c = -\frac{2}{n} \sum (y_i - w z_i - c) = -\frac{2}{n}(n\bar{y} - w \times 0 - nc) = 2(c - \bar{y})$. $\partial L / \partial w = -\frac{2}{n} \sum z_i (y_i - w z_i - c) = -\frac{2}{n}(nm - wn - c \times 0) = 2(w - m)$. Las dos derivadas se anulan para $w^* = m$ y $c^* = \bar{y} = 222{,}33$ k€. Aquí $m = 1\,637{,}5 / 22{,}796 \approx 71{,}83$ k€. Reconvertido con la ecuación 3.8: $a = m / \sigma = \operatorname{cov}(x, y)/\sigma_x^2 \approx 3{,}151$, la pendiente de la ecuación 3.9.

**c)** $w_{\text{nuevo}} = w - \eta \times 2(w - w^*)$, así que $w_{\text{nuevo}} - w^* = (1 - 2\eta)(w - w^*)$. El cálculo es el mismo para $c$. Con $\eta = 0{,}3$, el factor vale $1 - 0{,}6 = 0{,}4$. En el ejercicio 3.2, la derivada ya valía $2(b - \bar{r})$, de ahí el mismo factor.

**d)** Con $\eta = 0{,}5$, el factor vale 0: un solo paso lleva al mínimo. El descenso converge mientras $|1 - 2\eta| < 1$, es decir, $0 < \eta < 1$. Con $\eta = 1$, el factor vale −1: el ajuste rebota indefinidamente entre dos puntos. Más allá de 1, el descenso diverge.

**e)** Desarrollando el cuadrado y usando a): $L = \overline{y^2} + w^2 + c^2 - 2wm - 2c\bar{y} = (w - m)^2 + (c - \bar{y})^2 + \overline{y^2} - m^2 - \bar{y}^2$, donde $\overline{y^2}$ es la media de los cuadrados de los precios. El último término es el valor de $L$ en el mínimo: $54\,637{,}33 - 5\,159{,}86 - 49\,432{,}11 \approx 45{,}36$ k€². Una curva de nivel reúne los ajustes con la misma pérdida, es decir, con la misma distancia $(w - w^*)^2 + (c - c^*)^2$ al mínimo: es un círculo centrado en el mínimo. El gradiente apunta hacia el centro, de ahí el trayecto casi rectilíneo de la lección 3.5.

**f)** En m², $\sum x_i = 384 \neq 0$ y $\frac{1}{n}\sum x_i^2 \approx 4\,615{,}67 \neq 1$. El desarrollo de $L(a, b)$ contiene entonces un término cruzado $2 \times 64\, ab$, y $a^2$ pesa unas 4600 veces más que $b^2$. Las curvas de nivel son elipses muy alargadas e inclinadas: es el valle de la figura del libro.

### Comprobar en el notebook

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) suma de los z = {np.sum(zz):.1e}, suma de los z² = {np.sum(zz ** 2):.4f}')

In [ ]:
m = np.mean(zz * y)

def step(w, c, eta):
    """Devuelve (w, c) tras una actualización (ecuación 3.7 con z en lugar de x)."""
    e = y - (w * zz + c)
    return w + eta * 2 / len(x) * np.sum(zz * e), c + eta * 2 / len(x) * np.sum(e)

In [ ]:
# Comprobación de b) y d)
if m is None or step(1, 1, 0.1) is None:
    print('Completa la celda anterior.')
else:
    print(f'b) m = {m:.2f} →', 'correcto' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'revisar')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = step(w0, c0, 0.5)
    print(f'd) un paso con η = 0,5: (w ; c) = ({w1:.2f} ; {c1:.2f}) →',
          'correcto' if np.allclose([w1, c1], [m, y.mean()]) else 'revisar')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        sequence = []
        for _ in range(4):
            w, c = step(w, c, eta)
            sequence.append(round(c, 2))
        print(f'   η = {eta}: c tras 1, 2, 3, 4 pasos = {sequence}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLUE)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ por desviación típica)'); plt.ylabel('c (k€)'); plt.title('e) Círculos'); plt.show()